# Spider 1.0 Text-to-SQL QLoRA — Qwen3-4B + Gemma 3 4B

**Hardware:** works on Kaggle 2× Tesla T4, 1× GPU, Colab, or a local CUDA box (auto-detected)
**Training:** QLoRA with Unsloth, DDP when 2+ GPUs are available
**Dataset:** Spider 1.0 — 7,000 train / 1,034 validation examples, downloaded automatically from the Hugging Face Hub
**Models:** Qwen3-4B-Instruct and Gemma 3 4B-IT

This notebook trains **Qwen first**, uploads its adapter to Hugging Face, then trains **Gemma** and uploads its adapter.
It is designed to run start-to-finish with no manual path editing:

- The working directory, GPU count, and Hugging Face token are all detected automatically.
- All paths are derived from that detection — nothing is hardcoded to a single environment.
- Every long-running step (training, upload) is idempotent: re-running a cell after an interruption
  picks up where it left off instead of redoing finished work.
- The final evaluation section downloads the Spider SQLite databases dynamically (via `kagglehub`) and
  computes exact match, valid-SQL rate, and execution accuracy for all four systems.

**Before running:** create a Kaggle Secret (or environment variable, or be ready to paste one when prompted)
named `HF_TOKEN` with a Hugging Face **write** token.

## 1. Install the training stack

If Kaggle reports that core packages were replaced, restart the session and re-run from the top —
every step below is safe to re-run because it checks for already-completed work before redoing it.

In [ ]:
%%capture
!pip install -q -U "unsloth" "unsloth_zoo"
!pip install -q -U "transformers>=4.55,<5" "trl>=0.20,<1" "accelerate>=1.5,<2" "peft>=0.17,<1"
!pip install -q -U "bitsandbytes>=0.46.1" "datasets>=3.0,<5" "huggingface_hub>=0.30,<1" "sentencepiece"
!pip install -q -U "kagglehub>=0.3"


In [ ]:
# Quick import check so a broken install fails fast, with a clear message,
# instead of surfacing a confusing error three cells later.
missing = []
for pkg in ["torch", "transformers", "trl", "accelerate", "peft", "bitsandbytes", "datasets", "huggingface_hub"]:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    raise RuntimeError(
        f"These packages failed to import after installation: {missing}. "
        "Re-run the install cell above, and restart the kernel/session if Kaggle asked you to."
    )

print("All required packages import cleanly.")


## 2. Environment + GPU detection

Detects whether this is Kaggle, Colab, or a local machine, and picks a working directory accordingly.
It also detects however many GPUs are actually present (1 or 2+) instead of assuming exactly two —
training later adapts `--nproc_per_node` to whatever is found.

In [ ]:
import os
import torch

def detect_environment():
    if os.path.isdir("/kaggle/working"):
        return "kaggle"
    if "COLAB_GPU" in os.environ or os.path.isdir("/content"):
        return "colab"
    return "local"

ENV = detect_environment()

if ENV == "kaggle":
    WORK_DIR = "/kaggle/working"
elif ENV == "colab":
    WORK_DIR = "/content/spider_qlora_run"
else:
    WORK_DIR = os.path.join(os.getcwd(), "spider_qlora_run")

os.makedirs(WORK_DIR, exist_ok=True)
print("Detected environment:", ENV)
print("Working directory   :", WORK_DIR)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. On Kaggle: Settings -> Accelerator -> GPU T4 x2. "
        "On Colab: Runtime -> Change runtime type -> GPU."
    )

NUM_GPUS = torch.cuda.device_count()
print("\nGPUs detected:", NUM_GPUS)
for i in range(NUM_GPUS):
    print(" ", i, torch.cuda.get_device_name(i))

if NUM_GPUS < 2:
    print(
        "\nWARNING: only", NUM_GPUS, "GPU(s) found. This notebook was tuned for Kaggle's 2x T4, "
        "so it will still run correctly on 1 GPU, just roughly 2x slower. No changes needed."
    )


## 3. Hugging Face authentication + repositories

The token is picked up automatically, in order of preference:

1. A Kaggle Secret named `HF_TOKEN`
2. An `HF_TOKEN` environment variable
3. An interactive prompt (Colab / local), which never echoes or stores the token on disk

Both model repositories are created **before either training run starts**.

In [ ]:
from huggingface_hub import HfApi

def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass

    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    try:
        import getpass
        token = getpass.getpass("Enter your Hugging Face WRITE token (HF_TOKEN): ")
        if token:
            return token
    except Exception:
        pass

    return None

HF_TOKEN = get_hf_token()
if not HF_TOKEN:
    raise RuntimeError(
        "No Hugging Face token found. On Kaggle: Add-ons -> Secrets -> add HF_TOKEN. "
        "Elsewhere: set the HF_TOKEN environment variable, or re-run this cell and paste it when prompted."
    )

api = HfApi(token=HF_TOKEN)
who = api.whoami()
HF_USERNAME = who["name"]
print("Hugging Face account:", HF_USERNAME)

QWEN_REPO_ID = f"{HF_USERNAME}/qwen3-4b-spider-sql-lora"
GEMMA_REPO_ID = f"{HF_USERNAME}/gemma3-4b-spider-sql-lora"

for repo_id in [QWEN_REPO_ID, GEMMA_REPO_ID]:
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, token=HF_TOKEN)
    print("Repository ready:", repo_id)


## 4. Configuration

All paths are derived from `WORK_DIR` (set in section 2), so nothing here is hardcoded to a single
platform. Spider 1.0 has an official training split and validation split, so no synthetic
train/validation splitting logic is needed.

In [ ]:
SPIDER_TRAIN_FILE = os.path.join(WORK_DIR, "spider_train.jsonl")
SPIDER_VAL_FILE = os.path.join(WORK_DIR, "spider_validation.jsonl")

MAX_SEQ_LENGTH = 4096

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

SEED = 3407
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 1
WARMUP_RATIO = 0.05
LOGGING_STEPS = 25
EVAL_STEPS = 250
SAVE_TOTAL_LIMIT = 2

QWEN_BASE_MODEL = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
GEMMA_BASE_MODEL = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit"

QWEN_OUT = os.path.join(WORK_DIR, "qwen3-4b-spider-sql-lora")
GEMMA_OUT = os.path.join(WORK_DIR, "gemma3-4b-spider-sql-lora")
QWEN_CKPT = os.path.join(WORK_DIR, "qwen3-spider-checkpoints")
GEMMA_CKPT = os.path.join(WORK_DIR, "gemma3-spider-checkpoints")

for d in [QWEN_OUT, GEMMA_OUT, QWEN_CKPT, GEMMA_CKPT]:
    os.makedirs(d, exist_ok=True)

EFFECTIVE_BATCH_SIZE = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * max(NUM_GPUS, 1)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)
print("Qwen base model     :", QWEN_BASE_MODEL)
print("Gemma base model    :", GEMMA_BASE_MODEL)
print("Qwen adapter out    :", QWEN_OUT)
print("Gemma adapter out   :", GEMMA_OUT)


## 5. Load Spider 1.0

This conversion of Spider 1.0 contains the database schema + question in the user message and the
gold SQL in the assistant message. The download is retried automatically if the Hub has a transient
hiccup.

In [ ]:
from datasets import load_dataset

def load_spider(max_retries=3):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return load_dataset("lianghsun/spider-text2sql-bench")
        except Exception as e:
            last_err = e
            print(f"Download attempt {attempt}/{max_retries} failed: {e}")
    raise RuntimeError("Could not download the Spider dataset from the Hub.") from last_err


def resolve_split_names(dataset_dict):
    '''Different uploads of the same dataset don't always use the same split
    names ('validation' vs 'dev' vs 'val' vs 'test'). Detect them instead of
    hardcoding, so a rename on the Hub side doesn't break the notebook.'''
    available = list(dataset_dict.keys())

    train_name = next((s for s in available if "train" in s.lower()), None)

    val_priority = ["validation", "valid", "dev", "val", "test", "eval", "evaluation"]
    val_name = None
    for candidate in val_priority:
        for s in available:
            if candidate == s.lower():
                val_name = s
                break
        if val_name:
            break
    if val_name is None:
        # fall back to substring match, then to "the other split"
        for candidate in val_priority:
            val_name = next((s for s in available if candidate in s.lower()), None)
            if val_name:
                break
    if val_name is None:
        non_train = [s for s in available if s != train_name]
        val_name = non_train[0] if non_train else None

    if train_name is None or val_name is None:
        raise RuntimeError(
            f"Could not identify train/validation splits among {available}. "
            "Inspect `spider` manually and set SPIDER_TRAIN_SPLIT / SPIDER_VAL_SPLIT yourself."
        )
    return train_name, val_name


spider = load_spider()
print(spider)

SPIDER_TRAIN_SPLIT, SPIDER_VAL_SPLIT = resolve_split_names(spider)
print("\nResolved splits -> train:", SPIDER_TRAIN_SPLIT, "| validation:", SPIDER_VAL_SPLIT)

print("Train:", len(spider[SPIDER_TRAIN_SPLIT]))
print("Validation:", len(spider[SPIDER_VAL_SPLIT]))

if len(spider[SPIDER_TRAIN_SPLIT]) != 7000 or len(spider[SPIDER_VAL_SPLIT]) != 1034:
    print(
        "WARNING: split sizes differ from the expected Spider 1.0 counts (7000/1034). "
        "Continuing with whatever the Hub currently serves — this is informational, not fatal."
    )

print("\nExample:")
print(spider[SPIDER_TRAIN_SPLIT][0])


## 6. Save the exact train/validation data used by both runs

In [ ]:
import json

def write_jsonl(dataset, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in dataset:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

write_jsonl(spider[SPIDER_TRAIN_SPLIT], SPIDER_TRAIN_FILE)
write_jsonl(spider[SPIDER_VAL_SPLIT], SPIDER_VAL_FILE)

print("Train:", SPIDER_TRAIN_FILE, os.path.getsize(SPIDER_TRAIN_FILE), "bytes")
print("Validation:", SPIDER_VAL_FILE, os.path.getsize(SPIDER_VAL_FILE), "bytes")


## 7. Validate the training format

The model must receive schema + question and learn the corresponding SQL.

In [ ]:
with open(SPIDER_TRAIN_FILE, encoding="utf-8") as f:
    sample = json.loads(f.readline())

print(json.dumps(sample, indent=2, ensure_ascii=False)[:5000])

messages = sample["messages"]
assert any(m["role"] == "system" for m in messages), "Expected a system message with the schema."
assert any(m["role"] == "user" for m in messages), "Expected a user message with the question."
assert any(m["role"] == "assistant" for m in messages), "Expected an assistant message with the gold SQL."
assert next(m["content"] for m in messages if m["role"] == "assistant").strip(), "Assistant message is empty."

print("\nFormat check: PASS")


## 8. Write the training script

The notebook launches this script separately for Qwen and Gemma so failures are isolated.
It works unmodified on 1 GPU or 2+ GPUs, and it automatically resumes from the newest checkpoint
if a previous run of the same model was interrupted.

In [ ]:
TRAIN_SCRIPT = '\nimport os, glob, argparse, math\nos.environ.setdefault("NCCL_P2P_DISABLE", "1")\nos.environ.setdefault("NCCL_IB_DISABLE", "1")\n\nimport torch\nfrom datasets import load_dataset\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--base_model", required=True)\n    p.add_argument("--train_jsonl", required=True)\n    p.add_argument("--eval_jsonl", required=True)\n    p.add_argument("--output_dir", required=True)\n    p.add_argument("--checkpoint_dir", required=True)\n    p.add_argument("--max_seq_length", type=int, default=4096)\n    p.add_argument("--lora_r", type=int, default=16)\n    p.add_argument("--lora_alpha", type=int, default=32)\n    p.add_argument("--lora_dropout", type=float, default=0.05)\n    p.add_argument("--batch_size", type=int, default=1)\n    p.add_argument("--grad_accum", type=int, default=4)\n    p.add_argument("--lr", type=float, default=2e-4)\n    p.add_argument("--epochs", type=float, default=1)\n    p.add_argument("--warmup_ratio", type=float, default=0.05)\n    p.add_argument("--logging_steps", type=int, default=25)\n    p.add_argument("--eval_steps", type=int, default=250)\n    p.add_argument("--save_total_limit", type=int, default=2)\n    p.add_argument("--seed", type=int, default=3407)\n    args = p.parse_args()\n\n    os.makedirs(args.output_dir, exist_ok=True)\n    os.makedirs(args.checkpoint_dir, exist_ok=True)\n\n    local_rank = int(os.environ.get("LOCAL_RANK", 0))\n    rank = int(os.environ.get("RANK", 0))\n    world = int(os.environ.get("WORLD_SIZE", 1))\n    torch.cuda.set_device(local_rank)\n    main_rank = rank == 0\n\n    def log(x):\n        if main_rank:\n            print(x, flush=True)\n\n    major, _ = torch.cuda.get_device_capability(local_rank)\n    use_bf16 = major >= 8\n    use_fp16 = not use_bf16\n\n    from unsloth import FastModel\n    model, tokenizer = FastModel.from_pretrained(\n        model_name=args.base_model,\n        max_seq_length=args.max_seq_length,\n        load_in_4bit=True,\n        dtype=None,\n        full_finetuning=False,\n        device_map={"": local_rank},\n    )\n\n    model = FastModel.get_peft_model(\n        model,\n        r=args.lora_r,\n        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],\n        lora_alpha=args.lora_alpha,\n        lora_dropout=args.lora_dropout,\n        bias="none",\n        use_gradient_checkpointing="unsloth",\n        random_state=args.seed,\n        finetune_language_layers=True,\n        finetune_attention_modules=True,\n        finetune_mlp_modules=True,\n    )\n\n    train_ds = load_dataset("json", data_files=args.train_jsonl, split="train")\n    eval_ds = load_dataset("json", data_files=args.eval_jsonl, split="train")\n\n    def format_example(example):\n        return {"text": tokenizer.apply_chat_template(\n            example["messages"], tokenize=False, add_generation_prompt=False\n        )}\n\n    train_ds = train_ds.map(format_example, num_proc=1)\n    eval_ds = eval_ds.map(format_example, num_proc=1)\n\n    log(f"world={world} | train={len(train_ds)} | eval={len(eval_ds)}")\n    log(f"precision={\'BF16\' if use_bf16 else \'FP16\'}")\n\n    from trl import SFTTrainer, SFTConfig\n\n    steps_per_epoch = math.ceil(len(train_ds) / (args.batch_size * args.grad_accum * world))\n    total_steps = math.ceil(steps_per_epoch * args.epochs)\n    warmup_steps = max(1, round(total_steps * args.warmup_ratio))\n\n    config = SFTConfig(\n        output_dir=args.checkpoint_dir,\n        dataset_text_field="text",\n        max_length=args.max_seq_length,\n        per_device_train_batch_size=args.batch_size,\n        per_device_eval_batch_size=args.batch_size,\n        gradient_accumulation_steps=args.grad_accum,\n        num_train_epochs=args.epochs,\n        learning_rate=args.lr,\n        warmup_steps=warmup_steps,\n        lr_scheduler_type="cosine",\n        optim="paged_adamw_8bit",\n        weight_decay=0.01,\n        logging_steps=args.logging_steps,\n        eval_strategy="steps",\n        eval_steps=args.eval_steps,\n        save_strategy="steps",\n        save_steps=args.eval_steps,\n        save_total_limit=args.save_total_limit,\n        bf16=use_bf16,\n        fp16=use_fp16,\n        seed=args.seed,\n        report_to="none",\n        ddp_find_unused_parameters=False,\n        packing=False,\n    )\n\n    trainer = SFTTrainer(\n        model=model,\n        processing_class=tokenizer,\n        train_dataset=train_ds,\n        eval_dataset=eval_ds,\n        args=config,\n    )\n\n    resume_from_checkpoint = None\n    existing = sorted(\n        glob.glob(os.path.join(args.checkpoint_dir, "checkpoint-*")),\n        key=lambda p: int(p.rsplit("-", 1)[-1]),\n    )\n    if existing:\n        resume_from_checkpoint = existing[-1]\n        log(f"Resuming from checkpoint: {resume_from_checkpoint}")\n\n    trainer.train(resume_from_checkpoint=resume_from_checkpoint)\n\n    if main_rank:\n        model.save_pretrained(args.output_dir)\n        tokenizer.save_pretrained(args.output_dir)\n        print("Adapter saved:", args.output_dir, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'

QWEN_SCRIPT = os.path.join(WORK_DIR, "train_qwen.py")
GEMMA_SCRIPT = os.path.join(WORK_DIR, "train_gemma.py")

for path in [QWEN_SCRIPT, GEMMA_SCRIPT]:
    with open(path, "w", encoding="utf-8") as f:
        f.write(TRAIN_SCRIPT)

print(QWEN_SCRIPT)
print(GEMMA_SCRIPT)


## 9. Training launcher

Shared helper used for both Qwen and Gemma. It picks `--nproc_per_node` from the GPU count detected
in section 2, and it skips straight to "done" if a finished adapter already exists — so re-running
the notebook top-to-bottom after a successful run does not retrain anything by accident.

In [ ]:
import subprocess, shlex

def run_training(script_path, base_model, output_dir, checkpoint_dir, label):
    adapter_marker = os.path.join(output_dir, "adapter_config.json")
    if os.path.exists(adapter_marker):
        print(f"{label}: adapter already exists at {output_dir} — skipping training.")
        return

    nproc = max(NUM_GPUS, 1)
    cmd = [
        "torchrun", f"--nproc_per_node={nproc}", script_path,
        "--base_model", base_model,
        "--train_jsonl", SPIDER_TRAIN_FILE,
        "--eval_jsonl", SPIDER_VAL_FILE,
        "--output_dir", output_dir,
        "--checkpoint_dir", checkpoint_dir,
        "--max_seq_length", str(MAX_SEQ_LENGTH),
        "--lora_r", str(LORA_R),
        "--lora_alpha", str(LORA_ALPHA),
        "--lora_dropout", str(LORA_DROPOUT),
        "--batch_size", str(PER_DEVICE_TRAIN_BATCH_SIZE),
        "--grad_accum", str(GRADIENT_ACCUMULATION_STEPS),
        "--lr", str(LEARNING_RATE),
        "--epochs", str(NUM_TRAIN_EPOCHS),
        "--warmup_ratio", str(WARMUP_RATIO),
        "--logging_steps", str(LOGGING_STEPS),
        "--eval_steps", str(EVAL_STEPS),
        "--save_total_limit", str(SAVE_TOTAL_LIMIT),
        "--seed", str(SEED),
    ]

    print(f"\n=== {label} ({nproc} GPU process(es)) ===")
    print(" ".join(shlex.quote(x) for x in cmd))
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f"{label} training failed: exit code {result.returncode}")

    if not os.path.exists(adapter_marker):
        raise RuntimeError(f"{label} finished but no adapter was found at {output_dir}")
    print(f"{label}: training complete.")


## 10. Train Qwen3-4B first

All 7K Spider training examples are used. The 1,034 validation examples are never used for gradient
updates.

In [ ]:
run_training(QWEN_SCRIPT, QWEN_BASE_MODEL, QWEN_OUT, QWEN_CKPT, "Qwen3-4B")

## 11. Upload Qwen adapter

The Hugging Face repository was created in section 3. If the upload fails (e.g. a transient network
error), the adapter is still safe on disk and the notebook keeps going instead of crashing.

In [ ]:
from huggingface_hub import upload_folder

def upload_adapter(folder_path, repo_id, commit_message):
    if not os.path.exists(os.path.join(folder_path, "adapter_config.json")):
        raise RuntimeError(f"No adapter found at {folder_path}; train the model first.")
    try:
        upload_folder(
            folder_path=folder_path,
            repo_id=repo_id,
            repo_type="model",
            token=HF_TOKEN,
            commit_message=commit_message,
        )
        print(f"Uploaded: https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"WARNING: upload to {repo_id} failed ({e}).")
        print("The adapter is still saved locally at:", folder_path)

upload_adapter(QWEN_OUT, QWEN_REPO_ID, "Upload Qwen3-4B Spider 1.0 QLoRA adapter")


## 12. Train Gemma 3 4B

Same Spider data and QLoRA hyperparameters as Qwen, for a controlled comparison.

In [ ]:
run_training(GEMMA_SCRIPT, GEMMA_BASE_MODEL, GEMMA_OUT, GEMMA_CKPT, "Gemma3-4B")

## 13. Upload Gemma adapter

In [ ]:
upload_adapter(GEMMA_OUT, GEMMA_REPO_ID, "Upload Gemma 3 4B Spider 1.0 QLoRA adapter")

## 14. Qwen sanity check

Loads the saved adapter and generates one validation example. This only verifies that the saved
adapter loads and runs; it is not the final benchmark (see section 17).

In [ ]:
from unsloth import FastModel

qwen_model, qwen_tok = FastModel.from_pretrained(
    model_name=QWEN_OUT,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
FastModel.for_inference(qwen_model)

with open(SPIDER_VAL_FILE, encoding="utf-8") as f:
    ex = json.loads(f.readline())

prompt = ex["messages"][:-1]
inputs = qwen_tok.apply_chat_template(
    prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(qwen_model.device)

with torch.no_grad():
    out = qwen_model.generate(
        input_ids=inputs, max_new_tokens=256, do_sample=False,
        pad_token_id=qwen_tok.eos_token_id,
    )

generated = qwen_tok.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
print("GENERATED SQL:\n", generated)
print("\nGOLD SQL:\n", ex["messages"][-1]["content"])


## 15. Gemma sanity check

Loads the saved adapter and generates one validation example.

In [ ]:
gemma_model, gemma_tok = FastModel.from_pretrained(
    model_name=GEMMA_OUT,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
FastModel.for_inference(gemma_model)

with open(SPIDER_VAL_FILE, encoding="utf-8") as f:
    ex = json.loads(f.readline())

prompt = ex["messages"][:-1]
inputs = gemma_tok.apply_chat_template(
    prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(gemma_model.device)

with torch.no_grad():
    out = gemma_model.generate(
        input_ids=inputs, max_new_tokens=256, do_sample=False,
        pad_token_id=gemma_tok.eos_token_id,
    )

generated = gemma_tok.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
print("GENERATED SQL:\n", generated)
print("\nGOLD SQL:\n", ex["messages"][-1]["content"])


## 16. Download the Spider SQLite databases (for execution accuracy)

Execution accuracy requires running the generated SQL against the real Spider databases, not just
comparing strings. The databases are fetched dynamically — `kagglehub.dataset_download(...)` handles
the download and returns the local path, so there is nothing to configure by hand.

If the download is unavailable in your environment (e.g. no internet, or Kaggle dataset access
blocked), the notebook does **not** fail — it simply reports that execution accuracy is unavailable
and still reports exact match and valid-SQL rate for all four systems.

In [ ]:
def download_spider_databases():
    try:
        import kagglehub
    except ImportError:
        subprocess.run(["pip", "install", "-q", "kagglehub"], check=False)
        import kagglehub

    try:
        path = kagglehub.dataset_download("jeromeblanchet/yale-universitys-spider-10-nlp-dataset")
        print("Kaggle Spider dataset downloaded to:", path)
        return path
    except Exception as e:
        print("WARNING: could not download the Spider databases from Kaggle:", e)
        print("Execution accuracy will be reported as unavailable; the other two metrics still run.")
        return None


def build_db_index(root):
    index = {}
    if root is None:
        return index
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if fn.endswith(".sqlite") or fn.endswith(".db"):
                db_id = os.path.splitext(fn)[0]
                index.setdefault(db_id, os.path.join(dirpath, fn))
    return index


SPIDER_DB_ROOT = download_spider_databases()
DB_INDEX = build_db_index(SPIDER_DB_ROOT)
print("SQLite databases indexed:", len(DB_INDEX))


## 17. Final evaluation

Evaluates the same Spider validation examples for all four systems:

1. Qwen3-4B base
2. Qwen3-4B + LoRA
3. Gemma 3 4B base
4. Gemma 3 4B + LoRA

For each, this reports:

- **Exact match** — normalized string equality against the gold SQL
- **Valid SQL rate** — the query parses/plans successfully against its database (or, if the
  database can't be resolved, passes a basic SQL-completeness check)
- **Execution accuracy** — the query's result set matches the gold query's result set when both are
  run against the real Spider database (only computed for rows whose database could be resolved —
  see `execution_coverage` below)

Set `EVAL_SAMPLE_SIZE` to control how many validation rows are scored. Use `1034` for the full
official Spider dev set; the default below is smaller so the cell finishes quickly on Kaggle's T4s.

In [ ]:
import re
import sqlite3

EVAL_SAMPLE_SIZE = 200  # set to 1034 to evaluate the full official Spider validation set


def extract_db_id(example):
    if example.get("db_id"):
        return example["db_id"]
    for m in example.get("messages", []):
        if m.get("role") == "system":
            match = re.search(
                r"(?:database|db)[_ ]?(?:id|name)?\s*[:=]\s*([A-Za-z0-9_]+)",
                m.get("content", ""),
                re.IGNORECASE,
            )
            if match:
                return match.group(1)
    return None


def clean_sql(text):
    text = text.strip()
    text = re.sub(r"^```(sql)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()
    text = text.split(";")[0].strip()
    return text


def normalize_sql(sql):
    return re.sub(r"\s+", " ", sql.strip().lower()).rstrip(";")


def generate_sql(model, tokenizer, example, max_new_tokens=256):
    prompt = example["messages"][:-1]
    inputs = tokenizer.apply_chat_template(
        prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return clean_sql(tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))


def is_valid_sql(sql, db_path):
    if not sql:
        return False
    try:
        if db_path:
            con = sqlite3.connect(db_path)
            con.execute(f"EXPLAIN {sql}")
            con.close()
            return True
        return sqlite3.complete_statement(sql + ";")
    except Exception:
        return False


def execution_match(pred_sql, gold_sql, db_path):
    if not db_path:
        return None
    try:
        con = sqlite3.connect(db_path)
        con.execute("PRAGMA busy_timeout = 5000")
        pred_rows = set(map(tuple, con.execute(pred_sql).fetchall()))
        gold_rows = set(map(tuple, con.execute(gold_sql).fetchall()))
        con.close()
        return pred_rows == gold_rows
    except Exception:
        return False


def evaluate_system(model, tokenizer, examples, label):
    n = len(examples)
    exact = valid = exec_ok = exec_total = 0
    for ex in examples:
        gold_sql = ex["messages"][-1]["content"]
        pred_sql = generate_sql(model, tokenizer, ex)
        db_id = extract_db_id(ex)
        db_path = DB_INDEX.get(db_id) if db_id else None

        if normalize_sql(pred_sql) == normalize_sql(gold_sql):
            exact += 1
        if is_valid_sql(pred_sql, db_path):
            valid += 1
        match = execution_match(pred_sql, gold_sql, db_path)
        if match is not None:
            exec_total += 1
            if match:
                exec_ok += 1

    return {
        "system": label,
        "n": n,
        "exact_match": exact / n if n else 0.0,
        "valid_sql_rate": valid / n if n else 0.0,
        "execution_accuracy": (exec_ok / exec_total) if exec_total else None,
        "execution_coverage": exec_total / n if n else 0.0,
    }


with open(SPIDER_VAL_FILE, encoding="utf-8") as f:
    val_examples = [json.loads(line) for line in f]

eval_subset = val_examples[:EVAL_SAMPLE_SIZE]
print(f"Evaluating on {len(eval_subset)} of {len(val_examples)} validation examples.")

results = []

# --- Qwen base ---
qwen_base_model, qwen_base_tok = FastModel.from_pretrained(
    model_name=QWEN_BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True, dtype=None,
)
FastModel.for_inference(qwen_base_model)
results.append(evaluate_system(qwen_base_model, qwen_base_tok, eval_subset, "Qwen3-4B base"))
del qwen_base_model, qwen_base_tok
torch.cuda.empty_cache()

# --- Qwen + LoRA (reusing the model loaded in section 14) ---
results.append(evaluate_system(qwen_model, qwen_tok, eval_subset, "Qwen3-4B + LoRA"))

# --- Gemma base ---
gemma_base_model, gemma_base_tok = FastModel.from_pretrained(
    model_name=GEMMA_BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True, dtype=None,
)
FastModel.for_inference(gemma_base_model)
results.append(evaluate_system(gemma_base_model, gemma_base_tok, eval_subset, "Gemma3-4B base"))
del gemma_base_model, gemma_base_tok
torch.cuda.empty_cache()

# --- Gemma + LoRA (reusing the model loaded in section 15) ---
results.append(evaluate_system(gemma_model, gemma_tok, eval_subset, "Gemma3-4B + LoRA"))

print(f"\n{'System':<20}{'N':>6}{'ExactMatch':>12}{'ValidSQL':>10}{'ExecAcc':>10}{'ExecCov':>10}")
for r in results:
    exec_acc_str = f"{r['execution_accuracy']:.1%}" if r["execution_accuracy"] is not None else "n/a"
    print(
        f"{r['system']:<20}{r['n']:>6}{r['exact_match']:>12.1%}"
        f"{r['valid_sql_rate']:>10.1%}{exec_acc_str:>10}{r['execution_coverage']:>10.1%}"
    )

if not DB_INDEX:
    print(
        "\nNote: execution accuracy is 'n/a' because the Spider SQLite databases could not be "
        "resolved in this environment. Exact match and valid-SQL rate above are still fully valid."
    )


## 18. Run summary

In [ ]:
print("Qwen adapter :", f"https://huggingface.co/{QWEN_REPO_ID}")
print("Gemma adapter:", f"https://huggingface.co/{GEMMA_REPO_ID}")
print("Spider train      :", len(spider[SPIDER_TRAIN_SPLIT]))
print("Spider validation :", len(spider[SPIDER_VAL_SPLIT]))
print("Evaluated rows     :", len(eval_subset), "of", len(val_examples))
print("\nTraining, upload, and evaluation pipeline complete.")
